# Pretreatment Prevalence by Tumor Type (Supplementary Table S2)

Produces **Table S2**: for the same ICI-treated cohort as Table 1
(`11_demographic_table.ipynb`) and Tables S4/S5, cross-tabulates primary
tumor type against prior-treatment status (**Pretreated** vs **Not
Pretreated**) — i.e. whether the patient received systemic therapy before
the biopsy used for this study.

Tumor Type rows are nested exactly as in Table S4: the five broad
`cohort_broad` categories (SKCM/NSCLC/BLCA/CRC-MSI/**Other**), where
"Other" is a subtotal row spanning every patient in that bucket, followed
immediately by a granular breakdown of "Other" into its individual
`cohort_pan` tumour types (`assign_pan_cohorts()`, `3_meta.ipynb`). A
granular type gets its own row once it has at least
`SUBCOHORT_MIN_OTHER_TYPE_N` (`helpers/config.r`) ICI-treated patients;
smaller types are pooled into a residual **"Other (Other)"** row. A final
"All tumor types" row gives the grand total.

This table no longer stratifies by ICI Mechanism (it previously
cross-tabulated Tumor Type × ICI Mechanism, with a companion drug-regimen
legend) — Tumor Type is the only grouping axis now, matching the simpler
Tumor Type × [binary status] format used by Table S4. Table 1 still
reports ICI Mechanism as its own row for the cohort overall.

Each interior cell reports `n (row %)`, i.e. within each Tumor Type row,
the percentage of patients with known prior-treatment status who were
Pretreated vs Not Pretreated. Patients with unknown prior-treatment status
(`pretreated` is `NA` — `3_meta.ipynb` maps the raw "NULL" string to `NA`)
are excluded; the excluded count is reported in a footnote. The "Total"
column and "All tumor types" row are plain counts (no percentage), since
they are margins rather than within-row distributions. As with Tables
S4/S5, Other-bucket patients are counted in both the "Other" subtotal row
and their own granular child row — these are not meant to be summed
together; the "All tumor types" grand total is computed independently
from one row per patient.

**Input:** `cpi_ready.rds`
**Output:**
`figs/table_s2_pretreatment_by_subcohort.csv` -- kept purely as the intermediate
`assemble_supplementary_tables_excel.R` reads to build the combined
`supplementary_tables.xlsx`; no standalone `.html`/`.docx` are generated for
this table (see the combined workbook instead).

In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last() + full_cohort_name()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ── Load ICI-ready data ───────────────────────────────────────────────────────
# Same population as Table 1 and Tables S4/S5: ICI-treated patients with known
# RECIST response, one row per patient.
#
# IMPORTANT: cpi_ready.rds stores one row per (sampleId, gp) *and* a
# duplicated copy of every row with cohort_broad overwritten to
# "ICI-Treated" (added in 4_base.ipynb so Figure 3 can facet pooled-vs-per-
# cancer-type together). The gp filter below removes the first
# duplication; distinct() removes the second.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(gp == "T-cell", !is.na(response)) |>
  distinct(sampleId, .keep_all = TRUE)

In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# Tumor Type rows: identical construction to Table S4 -- the five
# cohort_broad categories plus a nested breakdown of "Other" into its
# granular cohort_pan types (>= SUBCOHORT_MIN_OTHER_TYPE_N patients get
# their own row; smaller types pool into "Other (Other)"). Shown as full
# tumor-type names rather than bare acronyms (full_cohort_name(),
# helpers/shortcuts.r), since this is a supplemental table rather than a
# space-constrained figure axis.
#
# Prior-treatment status: pretreated is numeric 1/0/NA (same field as
# Table 1's "Prior treatment" row). NA means unknown status (3_meta.ipynb
# maps the raw "NULL" string to NA) -- handled as a separate exclusion
# below, not folded into "Not Pretreated".
demo <- df_raw |>
  mutate(
    tumor_type_broad = fct_pin_last(factor(full_cohort_name(cohort_broad)), "Other"),
    pretreat_status   = factor(
      ifelse(pretreated == 1, "Pretreated", "Not Pretreated"),
      levels = c("Pretreated", "Not Pretreated")
    )
  )

other_subtype_counts <- demo |>
  filter(cohort_broad == "Other") |>
  count(cohort_pan, name = "n")

well_represented_other_types <- other_subtype_counts |>
  filter(n >= SUBCOHORT_MIN_OTHER_TYPE_N) |>
  pull(cohort_pan) |>
  as.character()

demo <- demo |>
  mutate(
    tumor_type_other_detail = case_when(
      cohort_broad != "Other" ~ NA_character_,
      as.character(cohort_pan) %in% well_represented_other_types ~
        paste0(full_cohort_name(as.character(cohort_pan)), " (Other)"),
      TRUE ~ "Other (Other)"
    ),
    tumor_type_other_detail = fct_pin_last(factor(tumor_type_other_detail), "Other (Other)")
  )

message("\"Other\" bucket breakdown (types with N >= ", SUBCOHORT_MIN_OTHER_TYPE_N, " shown individually):")
print(levels(demo$tumor_type_other_detail))

In [ ]:
# ── Build cross-tab: Tumor Type x Pretreatment Status, n (row %) per cell ─────
# Rows: the five broad cohorts (SKCM/NSCLC/BLCA/CRC-MSI/"Other" as a
# subtotal), followed by a nested breakdown of "Other", followed by an
# "All tumor types" grand total -- same subtotal-plus-detail-rows design as
# Table S4 (see Table S4's cell-4 for the full rationale).
#
# Interior cells report "n (row %)": the percentage of that Tumor Type's
# known-status patients who were Pretreated vs Not Pretreated. The "Total"
# column and "All tumor types" row are plain counts (table margins), not
# row percentages of themselves.
#
# Tumor Type is always known by construction (cohort_broad's case_when()
# falls back to "Other"); prior-treatment status can be genuinely unknown
# (pretreated is NA) -- that is the only source of exclusion here,
# diagnosed below.
known <- demo |> filter(!is.na(pretreat_status))

n_total    <- nrow(demo)
n_known    <- nrow(known)
n_excluded <- n_total - n_known
if (n_excluded > 0) {
  message(sprintf(
    "Note: %d of %d ICI-treated patients excluded from Table S2 (unknown prior-treatment status).",
    n_excluded, n_total
  ))
}

summarise_rows <- function(df, type_col) {
  df |>
    filter(!is.na({{ type_col }})) |>
    group_by(`Tumor Type` = {{ type_col }}, `Pretreatment` = pretreat_status) |>
    summarise(n = n(), .groups = "drop") |>
    group_by(`Tumor Type`) |>
    mutate(label = sprintf("%d (%d%%)", n, round(100 * n / sum(n)))) |>
    ungroup() |>
    select(`Tumor Type`, `Pretreatment`, label)
}

row_margins <- function(df, type_col) {
  df |>
    filter(!is.na({{ type_col }})) |>
    group_by(`Tumor Type` = {{ type_col }}) |>
    summarise(n = n(), .groups = "drop") |>
    mutate(`Pretreatment` = "Total", label = as.character(n)) |>
    select(`Tumor Type`, `Pretreatment`, label)
}

# Column margins and the grand total use `known` directly (one row per
# patient) so Other-bucket patients are counted once here, not twice --
# same reasoning as Table S4/S5's col_totals/grand_total.
other_known <- known |> filter(cohort_broad == "Other")

col_margins <- known |>
  group_by(`Pretreatment` = pretreat_status) |>
  summarise(n = n(), .groups = "drop") |>
  mutate(`Tumor Type` = "All tumor types", label = as.character(n)) |>
  select(`Tumor Type`, `Pretreatment`, label)

grand_total <- tibble(
  `Tumor Type` = "All tumor types", `Pretreatment` = "Total",
  label = as.character(nrow(known))
)

cell_summary <- bind_rows(
  summarise_rows(known,       tumor_type_broad),
  summarise_rows(other_known, tumor_type_other_detail)
)

row_totals <- bind_rows(
  row_margins(known,       tumor_type_broad),
  row_margins(other_known, tumor_type_other_detail)
)

# Row order: broad cohorts (freq order, "Other" pinned last as a
# subtotal), immediately followed by its granular children (freq order,
# "Other (Other)" pinned last), then the grand total. Column order:
# Pretreated, Not Pretreated, then the row-total column.
type_order <- c(
  setdiff(levels(demo$tumor_type_broad), "Other"),
  "Other",
  levels(demo$tumor_type_other_detail),
  "All tumor types"
)
pretreat_order <- c("Pretreated", "Not Pretreated", "Total")

# Split out of the main pipe since the final column-reordering step needs
# to inspect the pivoted frame's own column names -- the native R pipe
# (|>) doesn't support referring to the LHS from inside a nested call the
# way magrittr's %>% does (same issue documented in Table S4/S5's cell-4).
wide_pivoted <- bind_rows(cell_summary, row_totals, col_margins, grand_total) |>
  mutate(`Tumor Type` = factor(`Tumor Type`, levels = type_order)) |>
  arrange(`Tumor Type`) |>
  tidyr::pivot_wider(names_from = `Pretreatment`, values_from = label, values_fill = "\u2014")

wide <- wide_pivoted |>
  select(`Tumor Type`, all_of(intersect(pretreat_order, names(wide_pivoted))))

wide

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

footnote_text <- if (n_excluded > 0) {
  sprintf(
    paste0(
      "%d of %d ICI-treated patients (Table 1's cohort) are excluded here due to ",
      "unknown prior-treatment status. Interior cells report n (%% of that Tumor ",
      "Type's known-status patients); \"Total\"/\"All tumor types\" are plain ",
      "counts. \"Other\" is a subtotal spanning the rows below it; row/column ",
      "totals should not be summed with it."
    ),
    n_excluded, n_total
  )
} else {
  sprintf(
    paste0(
      "All %d ICI-treated patients (Table 1's cohort) are included. Interior ",
      "cells report n (%% of that Tumor Type's known-status patients); ",
      "\"Total\"/\"All tumor types\" are plain counts. \"Other\" is a subtotal ",
      "spanning the rows below it; row/column totals should not be summed with it."
    ),
    n_total
  )
}

# CSV — build intermediate only. Not a standalone deliverable: this
# table's actual output is its sheet in the combined
# supplementary_tables.xlsx (assemble_supplementary_tables_excel.R reads
# this .csv + the metadata sidecar below to build it).
wide |>
  readr::write_csv(paste0(out_dir, "table_s2_pretreatment_by_subcohort.csv"))

message("Saved: table_s2_pretreatment_by_subcohort.csv")

# Metadata sidecar (title + footnote) -- consumed by
# assemble_supplementary_tables_excel.R to build the combined supplemental
# tables Excel workbook without re-deriving this table's title/footnote
# text (footnote_text is computed above).
jsonlite::write_json(
  list(
    title    = "Supplementary Table S2. Pretreatment prevalence by tumor type",
    footnote = footnote_text
  ),
  paste0(out_dir, "table_s2_pretreatment_by_subcohort_meta.json"),
  auto_unbox = TRUE
)